In [31]:
# Module imports and auto-reload setup
%load_ext autoreload
%aimport if_lib, if_utils, if_dpp, if_graphics, if_consts, if_gc1dpp
%autoreload 1
import os
import json
import random

from if_utils import get_filename, show_data, save_traces

from if_lib import generate_random_challenge, read_HMAC, read_keypair, get_id_person, get_location_id, \
get_unit_id, get_resource_spec_id, get_resource, get_process, create_event, make_transfer, reduce_resource, set_user_location

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configuration and Endpoints

In [32]:
# Define constants for this use case
USE_CASE = 'ifusersflows'

# Zenflows API endpoint
ENDPOINT = 'https://proxy.dpp-test.dyne.im/zenflows/api'

# DPP service endpoint
DPP_URL = 'https://proxy.dpp-test.dyne.im/interfacer-dpp'

# All participants
USERS = ['designer1', 'designer2', 'service_prov1', 'service_prov2', 'manufacturer1', 'manufacturer2', 'customer1', 'customer2']

## File Path Configuration

In [33]:
# Calculate names of settings files
USERS_FILE = get_filename('cred_users.json', ENDPOINT, USE_CASE)
LOCS_FILE = get_filename('loc_users.json', ENDPOINT, USE_CASE)
UNITS_FILE = get_filename('units_data.json', ENDPOINT, USE_CASE)
SPECS_FILE = get_filename('res_spec_data.json', ENDPOINT, USE_CASE)
DPP_FILE = get_filename('dpp_data.json', ENDPOINT, USE_CASE)
RES_FILE = get_filename('initial_resources.json', ENDPOINT, USE_CASE)
PROCESS_FILE = get_filename('process_data.json', ENDPOINT, USE_CASE)

print(f"Data will be saved to:")
print(f"  Users: {USERS_FILE}")
print(f"  Locations: {LOCS_FILE}")
print(f"  Units: {UNITS_FILE}")
print(f"  Resource Specs: {SPECS_FILE}")
print(f"  DPP Data: {DPP_FILE}")
print(f"  Initial Resources: {RES_FILE}")
print(f"  Processes: {PROCESS_FILE}")

Data will be saved to:
  Users: use_cases/ifusersflows/proxy.dpp-test.dyne.im%2Fzenflows%2Fapi/cred_users.json
  Locations: use_cases/ifusersflows/proxy.dpp-test.dyne.im%2Fzenflows%2Fapi/loc_users.json
  Units: use_cases/ifusersflows/proxy.dpp-test.dyne.im%2Fzenflows%2Fapi/units_data.json
  Resource Specs: use_cases/ifusersflows/proxy.dpp-test.dyne.im%2Fzenflows%2Fapi/res_spec_data.json
  DPP Data: use_cases/ifusersflows/proxy.dpp-test.dyne.im%2Fzenflows%2Fapi/dpp_data.json
  Initial Resources: use_cases/ifusersflows/proxy.dpp-test.dyne.im%2Fzenflows%2Fapi/initial_resources.json
  Processes: use_cases/ifusersflows/proxy.dpp-test.dyne.im%2Fzenflows%2Fapi/process_data.json


## Initialize Data Structures

In [34]:
# Create data structures
process_data = {}
res_data = {}
event_seq = []
dpp_data = {}

# Initialize or load user data
if os.path.isfile(USERS_FILE):
    with open(USERS_FILE,'r') as f:
        users_data = json.loads(f.read())
    print("Credentials file available for users")
else:
    users_data = {}
    users_data['designer1'] = {
      "userChallenges": {
        "whereParentsMet": "London",
        "nameFirstPet": "Fuffy",
        "nameFirstTeacher": "Jim",
        "whereHomeTown": "Paris",
        "nameMotherMaid": "Wright"
      },
      "name": "Designer1",
      "username": "designer1_username",
      "email": "designer1@example.org",
      "note": "me.designer1.org"
    }
    users_data['designer2'] = {
      "userChallenges": {
        "whereParentsMet": "London",
        "nameFirstPet": "Fido",
        "nameFirstTeacher": "Mary",
        "whereHomeTown": "Amsterdam",
        "nameMotherMaid": "Wraight"
      },
      "name": "Designer2",
      "username": "designer2_username",
      "email": "designer2@example.org",
      "note": "me.designer2.org"
    }
    users_data['manufacturer1'] = {
        "userChallenges": {
            "whereParentsMet":"Amsterdam",
            "nameFirstPet":"Toby",
            "nameFirstTeacher":"Juliet",
            "whereHomeTown":"Rome",
            "nameMotherMaid":"Banks"
        },
        "name": "Manufacturer1",
        "username": "manufacturer1_username",
        "email": "manufacturer1@example.org",
        "note" : "me.manufacturer1.org"
    }
    users_data['manufacturer2'] = {
        "userChallenges": {
            "whereParentsMet":"Canicatti",
            "nameFirstPet":"Ula",
            "nameFirstTeacher":"Pugsly",
            "whereHomeTown":"Florence",
            "nameMotherMaid":"Ranks"
        },
        "name": "Manufacturer2",
        "username": "manufacturer2_username",
        "email": "manufacturer2@example.org",
        "note" : "me.manufacturer2.org"
    }
    users_data['service_prov1'] = {
        "userChallenges": {
            "whereParentsMet":"Amsterdam",
            "nameFirstPet":"Toby",
            "nameFirstTeacher":"Juliet",
            "whereHomeTown":"Rome",
            "nameMotherMaid":"Banks"
        },
        "name": "Service_prov1",
        "username": "service_prov1_username",
        "email": "service_prov1@example.org",
        "note" : "me.service_prov1.com"
    }
    users_data['service_prov2'] = {
        "userChallenges": {
            "whereParentsMet":"Canicatti",
            "nameFirstPet":"Ula",
            "nameFirstTeacher":"Pugsly",
            "whereHomeTown":"Florence",
            "nameMotherMaid":"Ranks"
        },
        "name": "service_prov2",
        "username": "service_prov2_username",
        "email": "service_prov2@example.org",
        "note" : "me.service_prov2.com"
    }
    users_data['customer1'] = {
        "userChallenges": {
            "whereParentsMet":"Rome",
            "nameFirstPet":"Ku",
            "nameFirstTeacher":"George",
            "whereHomeTown":"Florence",
            "nameMotherMaid":"Canti"
        },
        "name": "Customer1",
        "username": "customer1_username",
        "email": "customer1@example.org",
        "note" : "me.customer1.org"
    }
    users_data['customer2'] = {
        "userChallenges": {
            "whereParentsMet":"Rome",
            "nameFirstPet":"Ku",
            "nameFirstTeacher":"George",
            "whereHomeTown":"Florence",
            "nameMotherMaid":"Canti"
        },
        "name": "Customer2",
        "username": "customer2_username",
        "email": "customer2@example.org",
        "note" : "me.customer2.org"
    }
    with open(USERS_FILE,'w') as f:
        json.dump(users_data, f)

# Initialize or load location data
if os.path.isfile(LOCS_FILE):
    with open(LOCS_FILE,'r') as f:
        locs_data = json.loads(f.read())
    print("Location file available")
else:
    locs_data = {}
    locs_data['designer1'] = {
        "name": "Dyne",
        "lat": 52.39679,
        "long": 4.8781073,
        "addr": "Haparandadam 7, A1, 1013 AK Amsterdam, Netherlands",
        "note": "location.designer1.org"
    }
    locs_data['designer2'] = {
        "name": "Farback",
        "lat": 52.3767127,
        "long": 4.8990591,
        "addr": "Prins Hendrikkade 82 A, 1012 AE, Amsterdam, Netherlands",
        "note": "location.designer2.org"
    }
    locs_data['manufacturer1'] = {
        "name": "Fab Lab Hamburg",
        "lat" : 51.1531305,
        "long" : 5.2685045,
        "addr" : "Stockmeyerstr. 43 Halle 4K Eingang von der Wasserseite, 20457 Hamburg, Germany",
        "note": "location.manufacturer1.org"
    }
    locs_data['manufacturer2'] = {
        "name": "Fab Lab Amsterdam",
        "lat" : 52.372773,
        "long" : 4.8981243,
        "addr" : "Nieuwmarkt 4, 1012 CR Amsterdam, Netherlands",
        "note": "location.manufacturer2.org"
    }
    locs_data['service_prov1'] = {
        "name": "Bike Totaal B.V.",
        "lat" : 52.1971579,
        "long" : 5.3924199,
        "addr" : "Spaceshuttle 22, 3824 ML Amersfoort, Netherlands",
        "note": "location.service_prov1.com"
    }
    locs_data['service_prov2'] = {
        "name": "Lyondell Covestro Manufacturing",
        "lat" : 51.9650382,
        "long" : 4.0133683,
        "addr" : "Australiëweg 7, 3199 KB Maasvlakte Rotterdam, Netherlands",
        "note": "location.service_prov2.com"
    }
    locs_data['customer1'] = {
        "name": "CleanLease",
        "lat" : 51.47240440868687,
        "long" : 5.412460440524406,
        "addr" : "De schakel 30, 5651 Eindhoven, Netherlands",
        "note": "location.customer1.org"
    }
    locs_data['customer2'] = {
        "name": "OLVG",
        "lat" : 52.3585703,
        "long" : 4.9124307,
        "addr" : "Oosterpark 9, 1091 AC Amsterdam, Netherlands",
        "note": "location.customer2.org"
    }
    with open(LOCS_FILE,'w') as f:
        json.dump(locs_data, f)

# Initialize units and specs
if os.path.isfile(UNITS_FILE):
    with open(UNITS_FILE,'r') as f:
        units_data = json.loads(f.read())
    print(f"Unit file available")
else:
    units_data = {}

if os.path.isfile(SPECS_FILE):
    with open(SPECS_FILE,'r') as f:
        res_spec_data = json.loads(f.read())
    print(f"Resource Spec file available")
else:
    res_spec_data = {}

Credentials file available for users
Location file available
Unit file available
Resource Spec file available


## Authentication Setup: HMAC Generation

In [35]:
# Read HMAC or get it from the server
for user in USERS:
    read_HMAC(USERS_FILE, users_data, user, endpoint=ENDPOINT)

Server HMAC available for Designer1
Server HMAC available for Designer2
Server HMAC available for Service_prov1
Server HMAC available for service_prov2
Server HMAC available for Manufacturer1
Server HMAC available for Manufacturer2
Server HMAC available for Customer1
Server HMAC available for Customer2


## Cryptographic Key Generation

In [36]:
# Read the keypair
for user in USERS:
    read_keypair(USERS_FILE, users_data, user)

Keypair available for Designer1
Keypair available for Designer2
Keypair available for Service_prov1
Keypair available for service_prov2
Keypair available for Manufacturer1
Keypair available for Manufacturer2
Keypair available for Customer1
Keypair available for Customer2


## User Registration in Zenflows

In [37]:
# Read or get id of the person
for user in USERS:
    get_id_person(USERS_FILE, users_data, user, endpoint=ENDPOINT)

Id available for Designer1
Id available for Designer2
Id available for Service_prov1
Id available for service_prov2
Id available for Manufacturer1
Id available for Manufacturer2
Id available for Customer1
Id available for Customer2


## Location Registration and Assignment

In [38]:
# Read or get the location id
for user in USERS:
    get_location_id(LOCS_FILE, users_data[user], locs_data, user, endpoint=ENDPOINT)
    set_user_location(USERS_FILE, users_data, locs_data, user, endpoint=ENDPOINT)

Location id available for Dyne
Location id available for Designer1
Location id available for Farback
Location id available for Designer2
Location id available for Bike Totaal B.V.
Location id available for Service_prov1
Location id available for Lyondell Covestro Manufacturing
Location id available for service_prov2
Location id available for Fab Lab Hamburg
Location id available for Manufacturer1
Location id available for Fab Lab Amsterdam
Location id available for Manufacturer2
Location id available for CleanLease
Location id available for Customer1
Location id available for OLVG
Location id available for Customer2


## Unit of Measurement Registration

In [39]:
# Get the ids of all units
get_unit_id(UNITS_FILE, users_data['designer1'], units_data, 'piece', 'u_piece', 'om2:one', endpoint=ENDPOINT)
get_unit_id(UNITS_FILE, users_data['designer1'], units_data, 'mass', 'kg', 'om2:kilogram', endpoint=ENDPOINT)
get_unit_id(UNITS_FILE, users_data['designer1'], units_data, 'volume', 'lt', 'om2:litre', endpoint=ENDPOINT)
get_unit_id(UNITS_FILE, users_data['designer1'], units_data, 'time', 'h', 'om2:hour', endpoint=ENDPOINT)
get_unit_id(UNITS_FILE, users_data['designer1'], units_data, 'distance', 'km', 'om2:kilometer', endpoint=ENDPOINT)

Unit piece available
Unit mass available
Unit volume available
Unit time available
Unit distance available


## Process Definition

In [40]:
# Create the processes

# Create the process that delivers the service frame manufacturing
process_name = 'Create_bike_frame_manifacturing_service_ABC'
user_data = users_data['service_prov1']
note = f"Creation of bike frame manufacturing service ABC by {user_data['name']}"
get_process(process_name, process_data, note, user_data, endpoint=ENDPOINT)

# Create the process that delivers bike assembly service
process_name = 'Create_bike_assembly_service'
user_data = users_data['service_prov2']
note = f"Creation of bike assembly 1-hour service by {user_data['name']}"
get_process(process_name, process_data, note, user_data, endpoint=ENDPOINT)

# Create the process that deliver the service frame manufacturing
process_name = 'Create_aluminium_bike_frame'
user_data = users_data['manufacturer1']
note = f"Creation of aluminum bike frame by {user_data['name']}"
get_process(process_name, process_data, note, user_data, endpoint=ENDPOINT)

# Create the process that wraps creating design
process_name = 'Create_funky_bike_design'
user_data = users_data['designer1']
note = f"Creation of funky bike design by {user_data['name']}"
get_process(process_name, process_data, note, user_data, endpoint=ENDPOINT)

# Create the process that wraps creating the mirror
process_name = 'Creation_magic_bike_mirror'
user_data = users_data['manufacturer2']
note = f"Creation of magic bike mirror by {user_data['name']}"
get_process(process_name, process_data, note, user_data, endpoint=ENDPOINT)

# Create the process that wraps creating the bike
process_name = 'Creation_fancy_collaborative_bike'
user_data = users_data['designer2']
note = f"Creation of fancy collaborative bike by {user_data['name']}"
get_process(process_name, process_data, note, user_data, endpoint=ENDPOINT)

# Save process data to file
with open(PROCESS_FILE, 'w') as f:
    json.dump(process_data, f, indent=2)
print(f"Process data saved to {PROCESS_FILE}")

Process data saved to use_cases/ifusersflows/proxy.dpp-test.dyne.im%2Fzenflows%2Fapi/process_data.json


## Resource Specification Registration

In [41]:
# Read all the resource specifications
name = 'bike_manufacturing_service'
note = 'Specification bike assembly service'
classification = 'https://www.service_prov1.com/manufacture/bike'
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['service_prov1'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

name = 'aluminium'
note = 'Aluminium 10mt, 4cm pipes'
classification = 'https://www.wikidata.org/wiki/Q663'
default_unit_id = units_data['mass']['id']
get_resource_spec_id(SPECS_FILE, users_data['manufacturer1'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

name = 'aluminium_bike_frame'
note = 'Repository for aluminium bike frame'
classification = 'https://github.com/manufacturer1/aluminium_bike_frame'
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['manufacturer1'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

name = 'funky_bike_design'
note = 'Repository for funky bike design'
classification = 'https://github.com/designer1/funky_bike_design'
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['designer1'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

name = 'glass'
note = 'Glass for magic bike mirror'
classification = 'https://www.wikidata.org/wiki/Q11469'
default_unit_id = units_data['mass']['id']
get_resource_spec_id(SPECS_FILE, users_data['manufacturer2'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

name = 'magic_bike_mirror'
note = 'Repository for magic bike mirror'
classification = 'https://github.com/manufacturer2/magic_bike_mirror'
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['manufacturer2'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

name = 'bike_assembly_service'
note = 'Specification bike assembly service'
classification = 'https://www.service_prov2.com/assembly/bike'
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['service_prov2'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

name = 'fancy_collaborative_bike'
note = 'Repository for fancy collaborative bike'
classification = 'https://github.com/designer2/fancy_collaborative_bike'
default_unit_id = units_data['piece']['id']
get_resource_spec_id(SPECS_FILE, users_data['designer2'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

name = 'manufacturing_work'
note = 'Specification for manufacturing work'
classification = 'https://www.wikidata.org/wiki/Q187939'
default_unit_id = units_data['time']['id']
get_resource_spec_id(SPECS_FILE, users_data['service_prov1'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

name = 'assemblying_work'
note = 'Specification for assemblying work'
classification = 'https://www.wikidata.org/wiki/Q187939'
default_unit_id = units_data['time']['id']
get_resource_spec_id(SPECS_FILE, users_data['service_prov2'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

name = 'design_work'
note = 'Specification for design work'
classification = 'https://www.wikidata.org/wiki/Q82604'
default_unit_id = units_data['time']['id']
get_resource_spec_id(SPECS_FILE, users_data['designer1'], res_spec_data, name, note, classification, default_unit_id, endpoint=ENDPOINT)

Specification bike_manufacturing_service available
Specification aluminium available
Specification aluminium_bike_frame available
Specification funky_bike_design available
Specification glass available
Specification magic_bike_mirror available
Specification bike_assembly_service available
Specification fancy_collaborative_bike available
Specification manufacturing_work available
Specification assemblying_work available
Specification design_work available


## Initial Resource Creation

In [42]:
# We create the initial raw materials

res_name = 'aluminium'
amount = 5
get_resource(res_data, res_spec_data, res_name, users_data['manufacturer1'], event_seq, amount, endpoint=ENDPOINT)

res_name = 'glass'
amount = .2
get_resource(res_data, res_spec_data, res_name, users_data['manufacturer2'], event_seq, amount, endpoint=ENDPOINT)

# Save initial resources to file
with open(RES_FILE, 'w') as f:
    json.dump(res_data, f, indent=2)
print(f"Initial resources saved to {RES_FILE}")

Initial resources saved to use_cases/ifusersflows/proxy.dpp-test.dyne.im%2Fzenflows%2Fapi/initial_resources.json


## Component Production

Now we'll produce all the components needed for the bike assembly.

### Step 1: Produce Aluminium Bike Frame

In [43]:
# Produce aluminium bike frame
cur_res = action = event_note = amount = cur_pros = None
action = 'consume'
event_note='consume aluminium for bike frame'
amount = 5
cur_pros = process_data['Create_aluminium_bike_frame']
cur_res = res_data['aluminium']

event_id, ts = create_event(users_data['manufacturer1'], action, event_note, amount=amount, process=cur_pros, \
                 res_spec_data=res_spec_data, existing_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id':event_id, 'action' : action, 'res_name': cur_res['name'], 'res': cur_res['id']})
event_seq.append({'ts': ts, 'process_id':cur_pros['id'], 'name' : cur_pros['name']})

# Produce aluminum bike frame
action = 'produce'
event_note='produce aluminium bike frame'
amount = 1
res_data['aluminium_bike_frame'] = {
    "res_ref_id": f'aluminium_bike_frame-{random.randint(0, 10000)}',
    "name": 'aluminium bike frame',
    "spec_id": res_spec_data['aluminium_bike_frame']['id']
}
cur_res = res_data['aluminium_bike_frame']

event_id, ts = create_event(users_data['manufacturer1'], action, event_note, amount=amount, process=cur_pros, \
                 res_spec_data=res_spec_data, new_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id':event_id, 'action' : action, 'res_name': cur_res['name'], 'res': cur_res['id']})

print(f"✓ Aluminium bike frame produced (ID: {cur_res['id']})")

✓ Aluminium bike frame produced (ID: 06DE6X7AFC1J36PZDS49DHNMN0)


### Step 2: Transfer Frame to Designer2

In [44]:
# Transfer the aluminium bike frame from manufacturer1 to designer2
cur_res = action = event_note = amount = cur_pros = None
note='Transfer aluminium bike frame from manufacturer1 to designer2'
action = 'transfer'
amount = 1
cur_res = res_data['aluminium_bike_frame']

event_id, ts = make_transfer(users_data['manufacturer1'], action, note, users_data['designer2'], amount, cur_res, locs_data, res_spec_data, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id':event_id, 'action' : action, 'res_name': cur_res['name'], 'res': cur_res['id']})

print(f"✓ Frame transferred to designer2")

✓ Frame transferred to designer2


### Step 3: Produce Magic Bike Mirror

In [45]:
# Produce magic bike mirror
cur_res = action = event_note = amount = cur_pros = None
action = 'consume'
event_note='consume glass for bike mirror'
amount = .2
cur_pros = process_data['Creation_magic_bike_mirror']
cur_res = res_data['glass']

event_id, ts = create_event(users_data['manufacturer2'], action, event_note, amount=amount, process=cur_pros, \
                 res_spec_data=res_spec_data, existing_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id':event_id, 'action' : action, 'res_name': cur_res['name'], 'res': cur_res['id']})
event_seq.append({'ts': ts, 'process_id':cur_pros['id'], 'name' : cur_pros['name']})

action = 'produce'
event_note='produce magic bike mirror'
amount = 1
res_data['magic_bike_mirror'] = {
    "res_ref_id": f'magic_bike_mirror-{random.randint(0, 10000)}',
    "name": 'magic bike mirror',
    "spec_id": res_spec_data['magic_bike_mirror']['id']
}
cur_res = res_data['magic_bike_mirror']

event_id, ts = create_event(users_data['manufacturer2'], action, event_note, amount=amount, process=cur_pros, \
                 res_spec_data=res_spec_data, new_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id':event_id, 'action' : action, 'res_name': cur_res['name'], 'res': cur_res['id']})

print(f"✓ Magic bike mirror produced (ID: {cur_res['id']})")

✓ Magic bike mirror produced (ID: 06DE6X7E9R9EJ2G71G52W746MG)


### Step 4: Transfer Mirror to Designer2

In [46]:
# Transfer the mirror from manufacturer2 to designer2
cur_res = action = event_note = amount = cur_pros = None
note='Transfer magic bike mirror from manufacturer2 to designer2'
action = 'transfer'
amount = 1
cur_res = res_data['magic_bike_mirror']

event_id, ts = make_transfer(users_data['manufacturer2'], action, note, users_data['designer2'], amount, cur_res, locs_data, res_spec_data, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id':event_id, 'action' : action, 'res_name': cur_res['name'], 'res': cur_res['id']})

print(f"✓ Mirror transferred to designer2")

✓ Mirror transferred to designer2


### Step 5: Create Funky Bike Design

In [47]:
# Create funky bike design
cur_res = action = event_note = amount = cur_pros = None
action = 'work'
event_note='work to create funky bike design'
cur_pros = process_data['Create_funky_bike_design']
effort_spec = {}
effort_spec['unit_id'] = res_spec_data['design_work']['defaultUnit']
effort_spec['spec_id'] = res_spec_data['design_work']['id']
effort_spec['amount'] = 16

event_id, ts = create_event(users_data['designer1'], action, event_note, amount=0, process=cur_pros, \
                 res_spec_data=res_spec_data, effort_spec=effort_spec, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id':event_id, 'action' : action, 'amount': effort_spec['amount']})
event_seq.append({'ts': ts, 'process_id':cur_pros['id'], 'name' : cur_pros['name']})

action = 'produce'
event_note='produce design for funky bike'
amount = 1
res_data['funky_bike_design'] = {
    "res_ref_id": f'funky_bike_design-{random.randint(0, 10000)}',
    "name": 'funky bike design',
    "spec_id": res_spec_data['funky_bike_design']['id']
}
cur_res = res_data['funky_bike_design']

event_id, ts = create_event(users_data['designer1'], action, event_note, amount=amount, process=cur_pros, \
                 res_spec_data=res_spec_data, new_res=cur_res, endpoint=ENDPOINT)
event_seq.append({'ts': ts, 'event_id':event_id, 'action' : action, 'res_name': cur_res['name'], 'res': cur_res['id']})

print(f"✓ Funky bike design created (ID: {cur_res['id']})")

✓ Funky bike design created (ID: 06DE6X7HXPW3BD9VJ91DAKFMVC)


## Save All Component Data

Save all produced components to the resources file so they can be loaded in the production notebook.

In [48]:
# Update the resources file with all components
with open(RES_FILE, 'w') as f:
    json.dump(res_data, f, indent=2)
print(f"✓ All resources (raw materials + components) saved to {RES_FILE}")
print(f"  - {len(res_data)} resources total")

✓ All resources (raw materials + components) saved to use_cases/ifusersflows/proxy.dpp-test.dyne.im%2Fzenflows%2Fapi/initial_resources.json
  - 5 resources total


## Setup Complete - Summary

In [49]:
print("\n" + "="*60)
print("SETUP COMPLETE")
print("="*60)
print(f"\n✓ {len(users_data)} users registered")
print(f"✓ {len(locs_data)} locations created")
print(f"✓ {len(units_data)} units registered")
print(f"✓ {len(res_spec_data)} resource specifications created")
print(f"✓ {len(process_data)} processes defined")
print(f"✓ {len(res_data)} resources created (raw materials + components)")
print(f"\nAll data saved to JSON files in: use_cases/{USE_CASE}/")
print(f"\nComponents produced:")
print(f"  - Aluminium bike frame (transferred to designer2)")
print(f"  - Magic bike mirror (transferred to designer2)")
print(f"  - Funky bike design (created by designer1)")
print(f"\nYou can now run the production notebook to create the bike with DPP.")


SETUP COMPLETE

✓ 8 users registered
✓ 8 locations created
✓ 5 units registered
✓ 11 resource specifications created
✓ 6 processes defined
✓ 5 resources created (raw materials + components)

All data saved to JSON files in: use_cases/ifusersflows/

Components produced:
  - Aluminium bike frame (transferred to designer2)
  - Magic bike mirror (transferred to designer2)
  - Funky bike design (created by designer1)

You can now run the production notebook to create the bike with DPP.
